# 1. 합성곱 신경망 (CNN)

- 컴퓨터가 어떻게 이미지의 특징을 학습하고 분류하는지 알아볼 것
- 이를 위해, 이미지 분석에 특화된 `신경망` 인 CNN의 핵심 구성 요소를 먼저 알아 볼 것

## 1-1. 합성곱(Convolution) 연산

- 합성곱 연산은 이미지에서 **특징(feature)**을 추출하는 핵심 과정
- 여기서 특징이란, 이미지의 성격을 규정하는 **작은 패턴**(수직선, 수평선, 특정 색상, 질감 등)을 으 의미함.

### 1-1-1. 핵심 구성 요소

1. 입력 이미지
    - 컴퓨터에게 이미지는 숫자로 이루어진 격자(grid)에 불과함
    - 컬러 이미지라면, R, G, B 각 **채널**별로 3개의 숫자 격자가 겹쳐 있는 3차원 데이터
2. 필터 / 커널
    - 특징을 찾아내는 역할을 하는 아주 작은 행렬
    - `수직선`을 감지하는 필터는 수직 방향의 값들을 높게, 수평 방향의 값들은 낮게 할 **가중치**를 가지고 있음
    - 필터의 가중치 값 → 모델이 학습 과정에서 찾아내야 할 파라미터
3. 특징 맵
    - 필터가 이미지 전체를 훑고 지나간 결과물
    - 필터가 감지하려는 특징이 이미지의 특정 위치에 강하게 나타날수록, 해당 위치의 값이 커짐
        - **가중치(**수직선 필터의 수치) *** 특징** (원본 이미지의 수치)
    - 즉, **원본 이미지에서 특정 특징이 어디에 분포하는지**를 보여줌

### 1-1-2. 동작 원리

1. 필터가 입력 이미지의 좌측 상단부터 정해진 간격(stride)으로 이동하며 겹치는 영역을 체크
    - 파리 퇴치 알고리즘!?
    - 월말 평가 시험 문제!?
2. 각 위치에서 필터의 가중치와 이미지 픽셀 값을 **원소별로 곱한 후, 그 결과를 모두 합산**
    - 행렬 곱!?
3. 이 합산된 단일 값이 특징 맵의 한 픽셀을 이룸
4. 1~3 과정을 이미지 전체에 대해 반복 → 하나의 필터에 의한 하나의 특징 맵이 생성
- 만약 필터가 64개라면, 각각 다른 특징 (수평, 수직, 곡선, 특정 색상, 밝기 등…)을 감지하는 64개의 특징맵이 생성됨

### 1-1-3. 코드 구현

- pytorch의 `nn.Conv2d` : 2D 이미지에 대한 합성곱 계층
    1. **in_channels:** 입력 데이터의 채널 수(Red, Green, Blue, Alpha / HCV, HSB, HSL…. 등)
    2. **out_chnnels:** 사용할 필터의 개수. (생성될 특징 맵의 개수)
    3. **kernel_size:** 필터의 크기 (3 x 3 필터는 3)

In [ ]:
import torch.nn as nn

# 입력 채널 3개(RGB), 출력 채널 16개(16개의 필터 사용), 필터 크기 3x3, 스트라이드 1, 패딩 1
conv_layer = nn.Conv2d(in_channels=3, out_channels=16, kernel_size=3, stride=1, padding=1)

## 1-2. 풀링(Pooling) 연산

- 합성곱 연산을 통해 얻은 **특징 맵의 크기를 줄이는 과정**

### 1-2-1. 목적과 원리

1. Max Pooling 기준 동작 원리
    - 특징 맵을 정해진 크기 (예: 2 x 2)의 구역으로 나누고,
    - **각 구역에서 가장 큰 값 하나만 남기고** 나머지는 버리는 방식
2. 연산 효율성
    - 특징 맵의 가로, 세로 크기를 줄여서 다음 계층에서 처리해야 할 파라미터의 수를 감소시킴
3. 위치 불변성의 원리를 통한 강건성 확보
    - 풀링 윈도우 내에서 상하좌우 한 두 픽셀 정도 이동하더라도, 그 특징을 나타내는 **가장 큰 활성화 값**은 여전히 풀링 윈도우 안에 남아 있으므로, 연산의 결과로 **선택될 확률이 높음**
        - 이미지의 특정 특징이 **이미지 내에서 약간 이동**하더라도 그 특징을 일관되게 인식
    - 이미지의 노이즈나 사소한 위치에 과도하게 맞춰지는 과적합을 방지하는 효과

### 1-2-2. 코드 구현

- `nn.MaxPool2d`: 2d 최대 풀링 계층 정의

In [ ]:
# 2x2 크기의 윈도우로 최대 풀링을 수행
pool_layer = nn.MaxPool2d(kernel_size=2, stride=2)

## 1-3. 계층적 특징 추출

- 코드는 챕터 2에서!

### 1-3-1. 다중 합성곱 계층 활용하기

- 합성곱 층을 여러개 쌓아 올림으로써 더욱 복잡한 특징을 학습시킬 수 있게 됨
1. 예를 들어, 첫번째 층은 수직, 수평, 대각선 등을 학습하였다면, 그 출력 결과를 다음 층에게 전달
2. 두번째 층은 원본이미지가 아닌, 첫번째 층에서 넘겨받은 특징 맵을 입력으로 받음
3. 이제, 해당 특징 맵에서 각 기초적인 특징을 조합하여 복잡한 형태에 대한 탐색이 가능
    - 수평선과 수직선이 직교하는 부분은 **모서리**라는 특징으로 볼 수 있는 것
4. 만약, 여기서 더 깊게 층을 쌓는다면, **모서리와 원**과 같은 형태들을 모아 **눈 모양**과 같은 고수준 특징을 학습할 수 있음.

## 1-4. 완전 연결 계층

- 이미지의 특징을 모두 추출했다면, 추출된 특징을 종합하여, **최종적으로 이미지를 분류**하는 역할을 수행
- 이전 시간에 다뤘던 MLP 구조를 활용. 단. MLP를 활용하기 위해선, **평탄화** 작업이 필요

### 1-4-1. 동작 순서

1. **평탄화:** 풀링 계층을 통과한 다차원 특징맵을 1차원 벡터로 펼침
2. **분류**
3. **출력**

In [ ]:
import torch.nn as nn

# 간단한 분류기 예제
class SimpleClassifier(nn.Module):
    # num_classes: 분류할 클래스 수 (10가지 범주)
    def __init__(self, num_classes=10):
        super().__init__()

        # 3D 특징 맵을 1D 벡터로 변환
        self.flatten = nn.Flatten()

        # 이전 레이어에서 나온 특징 벡터의 크기를 입력으로 받음
            # in_features = 채널 수 * 특징 맵의 높이 * 특징 맵의 너비
                # 예: 64개의 5x5 특징맵 -> 64 * 5 * 5 = 1600
            # out_features = 은닉층의 뉴런 수 (128)
        self.fc1 = nn.Linear(in_features=1600, out_features=128)
        self.relu = nn.ReLU()
        # 최종 출력 레이어
            # out_features = 클래스 수 (num_classes)
        self.fc2 = nn.Linear(in_features=128, out_features=num_classes)

    def forward(self, x):
        x = self.flatten(x) # 3D 특징 맵 -> 1D 벡터
        x = self.fc1(x)
        x = self.relu(x)
        x = self.fc2(x) # 최종 클래스 점수(Logits) 출력
        return x